# Preprocessing of RGI 07 (Svalbard) point mass balance data

This notebook builds the Svalbard stake (point mass balance, PMB) dataset used by MassBalanceMachine.

**Pipeline**
1. Load the WGMS point mass balance data and keep stakes in Svalbard (`country == 'SJ'`).
2. Download OGGM glacier directories and sample topographic variables at each stake; drop stakes outside the RGI outline.
3. Merge stakes closer than 10 m (same glacier, year and period).
4. Add the sky-view factor (SVF) from the per-glacier grids made by `1.4 Create_grids` (part 1).
5. Re-assign stake IDs by elevation.
6. Clean measurement dates (drop summer balances, shift July starts to 1 August, drop winter balances ending in July).
7. Save `SJM_wgms_dataset_all.csv` to `cfg.dataPath/WGMS/Svalbard/csv/` and plot data overviews.

## Setting up:

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '../../../../')) # Add root of repo to import MBM

import pandas as pd
import warnings
import massbalancemachine as mbm
import pyproj
import matplotlib.pyplot as plt
import seaborn as sns
import xarray as xr
from cmcrameri import cm
from oggm import utils

# Region-specific paths (path_PMB_WGMS_csv, ...)
from papers.Transformer_TF.PMB_sources.RGI_07_Svalbard.scripts.config_SVA import *

# Helper functions shared across regions (OGGM, stake QC, SVF)
from papers.Transformer_TF.scripts.oggm import initialize_oggm_glacier_directories, export_oggm_grids, merge_pmb_with_oggm_data
from papers.Transformer_TF.scripts.pmb_processing import rename_stakes_by_elevation, check_point_ids_contain_glacier, remove_close_points, check_multiple_rgi_ids
from papers.Transformer_TF.scripts.pmb_processing import add_svf_from_rgi_zarr, plot_missing_svf_for_all_glaciers

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.SvalbardConfig()

mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

## Pre-processing:

### Load WGMS data:
- `mass_balance_point.csv`: point mass balance measurements (one row per stake and period).
- `glacier.csv`: glacier metadata, used to map the WGMS `glacier_id` to its RGI v6 ID (`rgi60_ids`).

In [ ]:
# Point mass balance measurements from WGMS
df_stakes = pd.read_csv(
    os.path.join(cfg.dataPath, 'WGMS/raw/mass_balance_point.csv'))

# Glacier metadata (WGMS id, RGI60 id, ...)
df_RGIId = pd.read_csv(cfg.dataPath + 'WGMS/raw/glacier.csv')

# Filter df_stakes to include only rows where country is Svalbard (SJ)
df_stakes = df_stakes[df_stakes['country'].isin(['SJ'])].reset_index(drop=True)

# Create a mapping dictionary from id to rgi60_ids
id_to_rgi_map = dict(zip(df_RGIId['id'], df_RGIId['rgi60_ids']))

# Add the RGIId column to the filtered DataFrame using glacier_id instead of id
# (glacier_id in df_stakes corresponds to id in df_RGIId)
df_stakes['RGIId'] = df_stakes['glacier_id'].map(id_to_rgi_map)

# Display glacier names with NaN RGIId
display(f"Number of rows with NaN RGIId: {df_stakes['RGIId'].isna().sum()}")
display(df_stakes[df_stakes['RGIId'].isna()]['glacier_name'].unique())

# Select and rename columns to the MBM naming convention
df_stakes_renamed = df_stakes.rename(
    columns={
        'point_id': 'POINT_ID',
        'latitude': 'POINT_LAT',
        'longitude': 'POINT_LON',
        'elevation': 'POINT_ELEVATION',
        'begin_date': 'FROM_DATE',
        'end_date': 'TO_DATE',
        'balance': 'POINT_BALANCE',
        'glacier_name': 'GLACIER',
        'year': 'YEAR',
        'country': 'COUNTRY',
        'balance_code': 'PERIOD'
    })

# Create new POINT_ID column: <glacier>_<year>_<WGMS measurement id>_<country>
df_stakes_renamed['POINT_ID'] = (df_stakes_renamed['GLACIER'] + '_' +
                                 df_stakes_renamed['YEAR'].astype(str) + '_' +
                                 df_stakes['id'].astype(str) + '_' +
                                 df_stakes_renamed['COUNTRY'])
# Only keep relevant columns in df
df_stakes_renamed = df_stakes_renamed[[
    'POINT_ID', 'POINT_LAT', 'POINT_LON', 'POINT_ELEVATION', 'FROM_DATE',
    'TO_DATE', 'POINT_BALANCE', 'GLACIER', 'PERIOD', 'RGIId', 'YEAR',
    'begin_date_unc', 'end_date_unc'
]]

# Remove rows with NaN values in POINT_LAT, POINT_LON, and POINT_ELEVATION
df_stakes_renamed = df_stakes_renamed.dropna(
    subset=['POINT_LAT', 'POINT_LON', 'POINT_ELEVATION'])

# change date format to YYYYMMDD
df_stakes_renamed['FROM_DATE'] = df_stakes_renamed['FROM_DATE'].astype(
    str).str.replace('-', '')
df_stakes_renamed['TO_DATE'] = df_stakes_renamed['TO_DATE'].astype(
    str).str.replace('-', '')

# Add data modification column to keep track of mannual changes
df_stakes_renamed['DATA_MODIFICATION'] = ''

# Print unique years
print("Unique years in the dataset:", df_stakes_renamed['YEAR'].unique())

display(df_stakes_renamed.head(2))

In [ ]:
# Check if any entry anywhere is NaN
display(df_stakes_renamed[df_stakes_renamed.isna().any(axis=1)])

In [ ]:
# Interactive map of the raw WGMS stakes, one layer per glacier
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium

m = plot_stakes_folium(df_stakes_renamed, color_map=None, layer_control=True)
m

### Add OGGM data:
Download the OGGM glacier directories (OGGM v1.6, `elev_bands_w_data`) of the stake glaciers, export their gridded data, and sample the topographic variables at the nearest grid cell of each stake. `merge_pmb_with_oggm_data` also flags whether each stake lies within its RGI outline (`within_glacier_shape`).

In [ ]:
df_stakes_renamed.columns

In [ ]:
rgi_ids_list = df_stakes_renamed['RGIId'].dropna().unique().tolist()

# initialize OGGM glacier directories
gdirs, rgidf = initialize_oggm_glacier_directories(
    cfg,
    rgi_region="07",
    rgi_version="62",
    base_url=
    "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
    log_level='WARNING',
    task_list=None,
    rgi_ids_list=rgi_ids_list)

# Save the OGGM gridded data of these glaciers to disk
export_oggm_grids(cfg, gdirs, rgi_region="07", subset_rgis=rgi_ids_list)

unique_rgis = df_stakes_renamed['RGIId'].unique()

# Sample topographic variables at each stake and flag stakes within the glacier outline
# (default variables: aspect, slope, topo, hugonnet_dhdt, consensus_ice_thickness, millan_v)
df_stakes_topo = merge_pmb_with_oggm_data(
    df_pmb=df_stakes_renamed,
    gdirs=gdirs,
    rgi_region="07",  # Svalbard
    rgi_version="62")

print("Unique years in the dataset:", df_stakes_topo['YEAR'].unique())

In [ ]:
# Restrict to within glacier shape
df_stakes_topo = df_stakes_topo[df_stakes_topo['within_glacier_shape'] == True]
df_stakes_topo = df_stakes_topo.drop(columns=['within_glacier_shape'])

# Display rows that have any NaN values
display(df_stakes_topo[df_stakes_topo.isna().any(axis=1)])

# Drop 3 rows where consensus_ice_thickness is NaN
#df_stakes_topo_dropped = df_stakes_topo.dropna(subset=['consensus_ice_thickness'])

print("Unique years in the dataset:", df_stakes_topo['YEAR'].unique())

In [ ]:
# Check for NaN
display(df_stakes_topo[df_stakes_topo.isna().any(axis=1)])

### Merge close stakes:
Within each glacier, year and period, stakes closer than 10 m are merged into one point with the mean `POINT_BALANCE` (see `remove_close_points`). Summer balances (and WGMS `index` balances) are removed first.

In [ ]:
from tqdm import tqdm

# Keep only annual and winter balances (drops summer and WGMS 'index' rows)
df_stakes_topo = df_stakes_topo[df_stakes_topo['PERIOD'].isin(['annual', 'winter'])]

# Merge stakes < 10 m apart, glacier by glacier
df_pmb_topo = pd.DataFrame()
for gl in tqdm(df_stakes_topo.GLACIER.unique(), desc='Merging stakes'):
    print(f'-- {gl.capitalize()}:')
    df_gl = df_stakes_topo[df_stakes_topo.GLACIER == gl]
    df_gl_cleaned = remove_close_points(df_gl)
    df_pmb_topo = pd.concat([df_pmb_topo, df_gl_cleaned])
# Drop temporary projected coordinates added by remove_close_points
df_pmb_topo.drop(['x', 'y'], axis=1, inplace=True)
df_pmb_topo.reset_index(inplace=True, drop=True)

print("Unique years in the dataset:", df_pmb_topo['YEAR'].unique())

### Add sky-view factor:
The SVF is sampled at each stake from the per-glacier grids (`<RGIId>.zarr` in `xr_masked_grids/`). Unlike the other regions, stakes on a masked (NaN) pixel are only plotted, not filled with the nearest valid pixel (`add_svf_nearest_valid` is not called).

In [ ]:
# Example of one svf file
rgi_id = df_pmb_topo.loc[0].RGIId

rgi_gl = "RGI60-07.00246"

# read ds with svf
path_masked_xr = os.path.join(cfg.dataPath,
                              'RGI_v6/RGI_07_Svalbard/xr_masked_grids/')

xr.open_zarr(path_masked_xr + f'{rgi_gl}.zarr').svf.plot()

In [ ]:
# Directory with the per-glacier SVF grids (<RGIId>.zarr)
path_masked_xr = os.path.join(cfg.dataPath,
                              "RGI_v6/RGI_07_Svalbard/xr_masked_grids")

# Nearest-pixel SVF sampling (NaN if the pixel is masked)
df_pmb_topo_svf = add_svf_from_rgi_zarr(
    df_pmb_topo,
    path_masked_xr,
    rgi_col="RGIId",
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    svf_var="svf",
    out_col="svf",
)
# Stakes that fell on a masked (NaN) SVF pixel. These are NOT filled below,
# so check this is 0 (it is for the current data)
df_missing = df_pmb_topo_svf[df_pmb_topo_svf["svf"].isna()].copy()
print("Missing SVF points:", len(df_missing))
print("Glaciers affected:", sorted(df_missing["RGIId"].unique()))

In [ ]:
# Plot SVF grids with missing-SVF stakes (red) and valid stakes (white) per glacier
plot_missing_svf_for_all_glaciers(
    df_with_svf=df_pmb_topo_svf,
    path_masked_xr=path_masked_xr,
    plot_valid_points=True,
    save_dir=
    None  # or e.g. os.path.join(cfg.dataPath, "diagnostics/svf_missing")
)

### Give new stake ids:
Stake IDs are re-assigned per glacier as `<GLACIER>_<rank>`, where the rank is the stake's mean elevation (0 = lowest).

In [ ]:
# Rename POINT_IDs to <GLACIER>_<rank>, ranked by elevation
df_pmb_new_ids = rename_stakes_by_elevation(df_pmb_topo_svf)

# Check that every POINT_ID contains its glacier name
check_point_ids_contain_glacier(df_pmb_new_ids)

print('Number of winter and annual samples:', len(df_pmb_new_ids))
print('Number of annual samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_new_ids[df_pmb_new_ids.PERIOD == 'winter']))

print("Unique years in the dataset:", df_pmb_new_ids['YEAR'].unique())

# Histogram of mass balance
df_pmb_new_ids['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

### Final cleaning:
Measurement periods are harmonised so that they match the hydrological year used by MBM:
1. Drop summer balances.
2. Shift annual and winter periods that start in July to 1 August.
3. Remove winter measurements that end in July.

Unlike US/CA and HMA, no filter on the start/end month combination or period length is applied.

In [ ]:
df_pmb_clean = df_pmb_new_ids.copy()

# drop summer period
df_pmb_clean = df_pmb_clean[df_pmb_clean["PERIOD"] != "summer"].copy()

# Ensure YYYYMMDD format
df_pmb_clean["FROM_DATE"] = df_pmb_clean["FROM_DATE"].astype(str).str.zfill(8)
df_pmb_clean["TO_DATE"] = df_pmb_clean["TO_DATE"].astype(str).str.zfill(8)

# Extract months (MM) of start and end of each measurement
df_pmb_clean["MONTH_START"] = df_pmb_clean["FROM_DATE"].str[4:6]
df_pmb_clean["MONTH_END"] = df_pmb_clean["TO_DATE"].str[4:6]

# Compute duration of each measurement period in days
df_pmb_clean["FROM_DATE_dt"] = pd.to_datetime(df_pmb_clean["FROM_DATE"],
                                              format="%Y%m%d",
                                              errors="coerce")
df_pmb_clean["TO_DATE_dt"] = pd.to_datetime(df_pmb_clean["TO_DATE"],
                                            format="%Y%m%d",
                                            errors="coerce")
df_pmb_clean["duration_days"] = (df_pmb_clean["TO_DATE_dt"] -
                                 df_pmb_clean["FROM_DATE_dt"]).dt.days


# Helper: print unique start/end months for winter and annual measurements
def print_months(df, label):
    winter = df[df.PERIOD == "winter"]
    annual = df[df.PERIOD == "annual"]
    print(f"\n{label}")
    print("Winter measurement months:")
    print("  Unique start months:", sorted(winter["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(winter["MONTH_END"].unique()))
    print("\nAnnual measurement months:")
    print("  Unique start months:", sorted(annual["MONTH_START"].unique()))
    print("  Unique end months:  ", sorted(annual["MONTH_END"].unique()))


# Helper: count and duration stats for each (start month, end month) combination
def print_combos(df, period, label):
    sub = df[df["PERIOD"] == period].copy()
    combo_counts = (sub.groupby(["MONTH_START", "MONTH_END"]).agg(
        count=("duration_days", "size"),
        mean_days=("duration_days", "mean"),
        min_days=("duration_days", "min"),
        max_days=("duration_days", "max")).reset_index().sort_values(
            ["MONTH_START", "MONTH_END"]))
    combo_counts["mean_days"] = combo_counts["mean_days"].round(0).astype(int)
    print(f"\n{label}")
    print(combo_counts.to_string(index=False))


# --- Before anything ---
print_months(df_pmb_clean, "Before any changes")
print_combos(df_pmb_clean, "annual",
             "Annual: all combinations before shifting")
print_combos(df_pmb_clean, "winter",
             "Winter: all combinations before shifting")

In [ ]:
# ======================
# 1) SHIFT JULY STARTS TO AUGUST 1st
# ======================
# July starts are moved to August 1st (start of the hydrological year)
# and the duration is recomputed
for period_type in ["annual", "winter"]:
    mask_july = ((df_pmb_clean["PERIOD"] == period_type) &
                 (df_pmb_clean["MONTH_START"] == "07"))
    n_july = int(mask_july.sum())
    df_pmb_clean.loc[mask_july, "FROM_DATE"] = (
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"].apply(
            lambda d: d.replace(month=8, day=1).strftime("%Y%m%d")))
    df_pmb_clean.loc[mask_july, "FROM_DATE_dt"] = pd.to_datetime(
        df_pmb_clean.loc[mask_july, "FROM_DATE"], format="%Y%m%d")
    df_pmb_clean.loc[mask_july, "duration_days"] = (
        df_pmb_clean.loc[mask_july, "TO_DATE_dt"] -
        df_pmb_clean.loc[mask_july, "FROM_DATE_dt"]).dt.days
    df_pmb_clean.loc[mask_july, "MONTH_START"] = "08"
    print(
        f"Shifted {n_july} {period_type} measurements from July to August 1st start"
    )

# ======================
# 2) REMOVE WINTER ENDING IN JULY
# ======================
# Winter measurements ending in July are removed (not relabelled as annual)
mask_winter_end_07 = (
    (df_pmb_clean["PERIOD"].astype(str).str.strip().str.lower() == "winter") &
    (df_pmb_clean["MONTH_END"] == "07"))
df_pmb_clean = df_pmb_clean.loc[~mask_winter_end_07].copy()

print_months(df_pmb_clean, "After shifting July starts to August")

## Save final dataframe:

In [ ]:
# Helper columns not needed in the final dataset
# (MONTH_START / MONTH_END are kept, as for the other European regions)
columns_to_drop = [
    'begin_date_unc', 'end_date_unc', 'FROM_DATE_dt', 'TO_DATE_dt',
    'duration_days'
]
df_pmb_clean = df_pmb_clean.drop(columns=columns_to_drop, errors='ignore')

# Save to csv
df_pmb_clean.to_csv(os.path.join(cfg.dataPath, path_PMB_WGMS_csv,
                                 'SJM_wgms_dataset_all.csv'),
                    index=False)

# Histogram of mass balance
df_pmb_clean['POINT_BALANCE'].hist(bins=20)
plt.xlabel('Mass balance [m w.e.]')

## Data overview:

In [ ]:
# Interactive map of all cleaned stakes, one layer per glacier
from papers.Transformer_TF.scripts.plotting.folium import plot_stakes_folium

m = plot_stakes_folium(df_pmb_clean, color_map=None, layer_control=True)
m

In [ ]:
print('Number of winter and annual samples:', len(df_pmb_clean))
print('Number of annual samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'annual']))
print('Number of winter samples:',
      len(df_pmb_clean[df_pmb_clean.PERIOD == 'winter']))

# Number of measurements per year (stacked annual/winter):
fig, axs = plt.subplots(2, 1, figsize=(20, 15))
ax = axs.flatten()[0]
df_pmb_clean.groupby(['YEAR', 'PERIOD']).size().unstack().plot(
    kind='bar',
    stacked=True,
    color=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
    ax=ax)
ax.set_title('Number of measurements per year for all glaciers')

# Total number of measurements per glacier
ax = axs.flatten()[1]
num_gl = df_pmb_clean.groupby(['GLACIER']).size().sort_values()
num_gl.plot(kind='bar', ax=ax)
ax.set_title('Number of total measurements per glacier since 1951')
plt.tight_layout()